# Taxi demand and operating patterns

**Daniel Umeadi — SQL + Python**

Where and when are qualified pickups concentrated, and does a calendar model beat weekly demand baselines?

Full January 2024 yellow taxi Parquet plus TLC zone lookup; 2,964,624 source rows. Training January 1–24; test January 25–31. Ten zones selected from training only.

## 1. Rebuild the analysis

Run the audited pipeline from this notebook directory. Source data is downloaded only when absent; hashes are checked before use.

In [1]:
import sys, subprocess
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
completed = subprocess.run([sys.executable, 'run.py'], check=True, capture_output=True, text=True)
print(completed.stdout)


{
  "source_rows": 2964624,
  "qualified_pickup_records": 2914490,
  "operational_records": 2848985,
  "top_training_zone": "JFK Airport",
  "holdout_zone_hours": 1680,
  "best_backtest_model": "last_week",
  "best_wape": 0.14624833749271485,
  "poisson_wape": 0.24578105286692292,
  "last_week_wape": 0.14624833749271485
}



## 2. Inspect the result and audit

Read denominators and exclusions before interpreting the findings.

In [2]:
summary=json.loads(Path('outputs/summary.json').read_text())
display(pd.Series(summary,name='value').to_frame())
checks=json.loads(Path('outputs/quality_checks.json').read_text())
display(pd.Series(checks,name='value').to_frame())

,value
source_rows,2964624
qualified_pickup_records,2914490
operational_records,2848985
top_training_zone,JFK Airport
holdout_zone_hours,1680
best_backtest_model,last_week
best_wape,0.146248
poisson_wape,0.245781
last_week_wape,0.146248


,value
source_rows,2964624
demand_rows,2914490
operational_rows,2848985
excluded_from_demand,50134
excluded_from_operations_after_demand,65505
outside_month_rows,18
unknown_or_nonphysical_pickup_rows,12018
nonpositive_or_missing_fare_rows,38341
sql_pandas_demand_reconcile,True
zone_dimension_unique,True


## 3. Inspect the analytical output

In [4]:
display(pd.read_csv('outputs/backtest_metrics.csv'))


,model,mae_records_per_zone_hour,rmse,wape,signed_bias_share
0,last_week,23.301190,37.112009,0.146248,0.000605
1,weekday_hour_mean,26.975397,40.457954,0.169309,-0.037186
2,poisson,39.159359,55.749042,0.245781,-0.037184


## 4. Interpretation

2,914,490 qualifying pickup records. Previous-week WAPE is 14.62%, versus 24.58% for the Poisson model. The simple baseline wins on this holdout; report that result rather than claiming model superiority.

**Limits:** No unique trip key, no idle-time or supply information, no unmet-demand measure. One month cannot establish yearly seasonality. Cash tips are absent. Analyst-selected quality thresholds affect the operational subset.

**Next decision:** Where and when are qualified pickups concentrated, and does a calendar model beat weekly demand baselines? 

## Charts

![01_zones](outputs/01_zones.png)

![02_heatmap](outputs/02_heatmap.png)

![03_forecast](outputs/03_forecast.png)

![04_comparison](outputs/04_comparison.png)

